In [ ]:
!pip install -q -U open_clip_torch

In [2]:
import os
import open_clip
import glob
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm

In [ ]:
import open_clip
print(open_clip.list_models())
print(open_clip.list_pretrained())

# Parse Data Path

In [3]:
keyframes_dir = "/kaggle/input/keyframes-extracted-data/keyframes_extracted"
all_keyframe_paths = dict()
for part in sorted(os.listdir(keyframes_dir)):
    data_part = part.replace('_extract','') # L21_a, L22_a,..
    all_keyframe_paths[data_part] = dict()

    data_part_path = os.path.join(keyframes_dir, part)
    video_dirs = sorted(os.listdir(data_part_path))
    video_ids = [video_dir.split('_')[-1] for video_dir in video_dirs]
    for video_id, video_dir in zip(video_ids, video_dirs):
        keyframe_paths = sorted(glob.glob(f'{data_part_path}/{video_dir}/*.jpg'))
        all_keyframe_paths[data_part][video_id] = keyframe_paths

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-gopt-16-SigLIP2-384',
    pretrained='webli', 
    device=device
)
model = model.to(device)
model.eval()

# Inference

In [ ]:
bs = 4
save_dir = './OpenCLIP_features'
if not os.path.exists(save_dir):
  os.mkdir(save_dir)

for key, video_keyframe_paths in tqdm(all_keyframe_paths.items()):
    video_ids = sorted(video_keyframe_paths.keys())
    
    if not os.path.exists(os.path.join(save_dir, key)):
        os.mkdir(os.path.join(save_dir, key))
    
    for video_id in tqdm(video_ids):
        video_feats = []
        video_keyframe_path = video_keyframe_paths[video_id]
        for i in range(0, len(video_keyframe_path), bs):
            # Support batchsize inferencing
            images = []
            image_paths = video_keyframe_path[i:i+bs]
            for image_path in image_paths:
                image = preprocess(Image.open(image_path)).unsqueeze(0)
                images.append(image)
            images = torch.cat(images).to(device)

            with torch.no_grad(), torch.cuda.amp.autocast():
                image_feats = model.encode_image(images)
            image_feats /= image_feats.norm(dim=-1, keepdim=True)

            for b in range(image_feats.shape[0]):
                video_feats.append(image_feats[b].detach().cpu().numpy().astype(np.float32).flatten())
        
        np.save(f'{save_dir}/{key}/{video_id}.npy', video_feats)

# Faiss index

In [ ]:
feature_path = '/kaggle/working/OpenCLIP_features/L21_a/V003.npy'
features = np.load(feature_path)

# In ra shape của mảng
print("Shape:", features.shape[1])

In [ ]:
features_dir = './OpenCLIP_features'

index = faiss.IndexFlatIP(features.shape[1])

for data_part in tqdm(sorted(os.listdir(features_dir))):
    for feature_path in tqdm(sorted(glob.glob(os.path.join(features_dir, data_part) +'/*.npy'))):
        feats = np.load(feature_path)
        for feat in feats:
            feat = feat.astype(np.float32).reshape(1,-1)
            index.add(feat)

faiss.write_index(index, f"SigLIP_cosine.bin")